# Industry Spillover Risk Pipeline — Week 3 Task 2: VAR Connectedness Analysis

**Markets:** China (CSI All-Share), United States (S&P 500 GICS)  
**Volatility input:** GARCH(1,1)-t conditional volatilities only  
**VAR lag orders:** China p=2, US p=8 (AIC-optimal, fixed)  
**GFEVD:** KPPS Generalised (order-invariant), H=10  
**Rolling window:** 200 trading days, step=1  
**Robustness:** common p ∈ {4, 8} re-estimation

## Glossary / Semantics

- <span style="color:#FF69B4">Lag order (p):</span>

    In time series analysis, $p$ (lag order) is the number of past days' data the model looks back on to forecast today's volatility. A lag of $p$=2 means today's sector volatility depends on what happened 1 and 2 days ago.
    The settings are different because they are selected mathematically by minimizing the AIC (`Akaike Information Criterion`). The US stock market incorporates information over a slightly longer historical horizon (`8` trading days) compared to the Chinese market (`2` trading days), reflecting distinct institutional structures and informational efficiency profiles.

- <span style="color:#FF69B4">Forecast horizon H=10:</span>

    $H$ represents how far into the future we project the system to see how shocks propagate. An $H=10$ setting evaluates the impact of an industry volatility shock over a standard 2-week financial horizon (10 trading days), following the canonical Diebold-Yilmaz (2014) setup. 

- <span style="color:#FF69B4">Rolling window 200 days and step = 1:</span>

    A 200-day rolling window means that instead of calculating one massive average for the entire 15-year dataset, the model takes a slice of 200 trading days (roughly 9-10 calendar months) to capture a snapshot of the market.

    Step = 1 means the window moves forward by exactly 1 trading day at a time, dropping the oldest day and adding the newest day. This generates a smooth, highly responsive daily time series of risk spillovers over time.

- <span style="color:#FF69B4">GARCH Fitting (INNOVATION_DIST='t'):</span>

    (Required by weekly plan) to accommodate fat tails (extreme market shocks), ensuring stable convergence when returns are rescaled by 100

- <span style="color:#FF69B4">ADF (Augmented Dickey-Fuller):</span>

    A statistical test used to verify if a time series is stationary (meaning its statistical properties like mean and variance don't change over time). Volatilities must pass this test before going into a VAR model

- <span style="color:#FF69B4">VAR / GFEVD:</span>

    VAR (Vector Autoregression) captures the mathematical relationships between all 11 sectors simultaneously. GFEVD (Generalized Forecast Error Variance Decomposition) breaks down the forecast error variance of sector A to see what percentage of it was caused by an unexpected shock originating in sector B. It is "Generalized" because it is independent of the order in which you input the sectors.

- <span style="color:#FF69B4">Robustness Lags:</span>

    Re-runs the models at fixed benchmarks ($p=4$ and $p=8$) to prove that the baseline results are structurally sound and not an artifact of choosing one specific lag

- <span style="color:#FF69B4">Rolling Connectedness:</span>

    Calculating the spillover matrix repeatedly across the moving 200-day windows to capture how risk networks evolve during crashes vs. bull markets.

- <span style="color:#FF69B4">Top N Edges (TOP_N_CONNS):</span>

    Filters out weak, insignificant relationships to keep only the strongest paths of risk transmission, preventing visual clutter in the network graph.

## Data Arrays & Code Variables

- <span style="color:#40E0D0">all_prices:</span>

    Raw stock market closing prices for the 11 sectors

- <span style="color:#40E0D0">all_returns:</span>

    Raw daily log retuns calculated as $\ln(P_t / P_{t-1})$

- <span style="color:#40E0D0">pct_returns:</span>

    The identical return data scaled up by multiplying by 100 to help the GARCH optimizer converge normally

## GARCH Mathematics

#### Core parameters estimated by standard GARCH(1,1) model to track daily variance ($\sigma^2_t$):

$$\sigma^2_t = \omega + \alpha \epsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

- $\omega$ <span style="color:#FFA07A">(Omega):</span>

    The baseline, long-term constant variance if no new shocks occur.

- $\alpha$ <span style="color:#FFA07A">(Alpha):</span>

    The "shock" parameter. It dictates how violently today's volatility reacts to a sudden market event yesterday ($\epsilon^2_{t-1}$).

- $\beta$ <span style="color:#FFA07A">(Beta):</span>

    The "persistence" parameter. It dictates how much of yesterday's volatility carries over into today ($\sigma^2_{t-1}$). A high $\beta$ means market anxiety lingers for a long time.

#### Volatility panel vs. GARCH panel:

- <span style="color:#FFA07A">Volatility panel:</span>

    Refers to the analysis of time-varying financial or economic uncertainty across multiple subjects (e.g., countries, firms, or asset classes) over time. It combines panel data (multiple entities observed over time) with volatility modeling (which estimates fluctuating variances in data, like stock returns or exchange rates).

- <span style="color:#FFA07A">GARCH panel:</span>

    A specific, highly rigorous type of volatility panel where the entries have been explicitly calculated using univariate GARCH filtering on asset returns to isolate conditional variances over time.

## Model Selection & Network Metrics

- AIC, BIC, FPE, and HQIC significance:

    These are Information Criteria metrics used to figure out the perfect lag order ($p$) for a VAR model. They score models based on how well they fit the data while penalizing them for using too many variables (to prevent overfitting).

    - <span style="color:#90EE90">AIC (Akaike Information Criterion)</span> & <span style="color:#90EE90">FPE (Final Prediction Error):</span>
    
        Tend to prefer larger, more flexible lag profiles.
    
    - <span style="color:#90EE90">BIC (Bayesian Information Criterion)</span> & <span style="color:#90EE90">HQIC (Hannan-Quinn Information Criterion):</span>

        Apply stricter mathematical penalties and prefer smaller, simpler lag profiles.

- Significance of <span style="color:#90EE90">rolling results<span>:

    Sequence of connectedness matrices and indices generated across the moving timeline. They signify how financial contagion transforms dynamically over time, identifying when corss-sector links tighten during economic stress.

- What is <span style="color:#90EE90">TSI</span>:

    Total Spillover Index. It acts as a single master percentage tracking how interconnected the entire stock market is at any given moment.  
    
    <span style="color:#90EE90">Significance:<span> 
    
    A low TSI indicates sectors are moving independently; a high TSI signifies high market stress where a shock to one industry spreads like a wildfire to all others.
    
    <span style="color:#90EE90">Computation:<span> 
    
    It is calculated using the GFEVD matrix by summing all off-diagonal values (the risk sectors transmit to others) and dividing it by the grand sum of the entire matrix (total system variation).

## 0.0 — Setup

Run this in terminal before importing:

```bash
pip install numpy ipykernel
pip install pandas ipykernel
pip install statsmodels ipykernel
pip install arch ipykernel
pip install openpyxl
```

## 1.0 — Imports, Logging & Global Hyperparameters

### Code Explanation:

<span style="color:#63C2FF">1.1 Imports & Warnings:</span>

- Imports, warning suppressions, and logging configuration
- Wires up the libraries used across the spillover pipeline:
    - numpy & pandas: data handling
    - arch: GARCH & EGARCH volatility models
    - statsmodels: VAR & ADF analysis (Augmented Dickey Fuller, is a common statistical test used to test whether a given Time series is stationary or not)
- Warnings are silenced globally since arch_model() raises noisy optimizer warnings that are already captured and logged via _log_issue() instead of being surfaced as Python warnings
- NFO-level logging: every pipeline stage logs a timestamped status line to the notebook output, which doubles as a lightweight audit trail of each run.

---

<span style="color:#63C2FF">1.2 Directory Layout:</span>

Resolve the project's directory layout relative to the notebook's location and make sure the `outputs/` and `logs/` directories exist before any later stage attempts to write a file into them.

---

<span style="color:#63C2FF">1.3 Hyperparameters:</span>

Global hyperparameters shared by these stages:
- GARCH/EGARCH fitting
- ADF stationarity testing
- VAR lag selection
- Diebold-Yilmaz GFEVD / rolling-connectedness analysis

Centralising them here means switching MARKET (or any threshold) reruns the whole pipeline consistently.

---

<span style="color:#63C2FF">1.4 Market File Configurations:</span>

Per-market schema configuration: maps each raw data file's column names and raw sector codes/labels onto the canonical GICS_SECTORS names used by every downstream stage. 

### Code Cell:

In [ ]:
# ── 1.1 — Imports & Warnings ──────────────────────────────────────────────────
import os, pickle, logging, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from arch import arch_model
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger(__name__)

# ── 1.2 — Directory layout ────────────────────────────────────────────────────
BASE_DIR    = Path().resolve()
DATA_DIR    = BASE_DIR / 'data'
OUTPUTS_DIR = BASE_DIR / 'outputs'
LOGS_DIR    = BASE_DIR / 'logs'
ISSUE_LOG   = LOGS_DIR / 'issue_log.txt'
for _d in [OUTPUTS_DIR, LOGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── 1.3 — Hyperparameters ─────────────────────────────────────────────────────
MARKETS          = ['china', 'us']

# GARCH fitting
INNOVATION_DIST  = 't'          # Student-t innovations
MAX_ITER         = 200
MAX_ITER_RETRY   = 500
PERSISTENCE_LOW  = 0.95
PERSISTENCE_HIGH = 0.99
JUMP_THRESHOLD   = 20.0         # flag |R| > 20 %

# ADF
ADF_MAXLAG       = 12
ADF_PVAL_THR     = 0.05

# VAR / GFEVD
MAX_LAGS         = 10           # AIC search bound
HORIZON          = 10           # GFEVD forecast horizon (DY 2014)

# Per-market fixed lag orders (AIC-optimal, pre-determined)
VAR_LAG_ORDERS   = {'china': 2, 'us': 8}

# Robustness check common lag orders
ROBUSTNESS_LAGS  = [4, 8]

# Rolling connectedness
ROLLING_WINDOW   = 200          # trading-day window
ROLLING_STEP     = 1

# GNN adjacency export
TOP_N_CONNS      = 5            # top edges per window for network snapshots

# ── GICS sector order ─────────────────────────────────────────────────────────
GICS_SECTORS = [
    'Energy', 'Materials', 'Industrials',
    'Consumer Discretionary', 'Consumer Staples', 'Health Care',
    'Financials', 'Information Technology', 'Communication Services',
    'Utilities', 'Real Estate',
]

# ── 1.4 — Market File Configurations ──────────────────────────────────────────
MARKET_CONFIGS = {
    'china': {
        'file'         : 'China Industry Index.xlsx',
        'date_col'     : 'Date',
        'price_col'    : 'Closeprice',
        'format'       : 'long',        # long: rows keyed by (Date, Index)
        'index_col'    : 'Index',       # column holding numeric sector codes
        'benchmark_col': 300,           # numeric code for the CSI 300 – drop it
        # Numeric sector code → GICS name
        'sector_map': {
            986    : 'Energy',
            987    : 'Materials',
            988    : 'Industrials',
            989    : 'Consumer Discretionary',
            990    : 'Consumer Staples',
            991    : 'Health Care',
            993    : 'Information Technology',
            994    : 'Communication Services',
            995    : 'Utilities',
            931775 : 'Real Estate',
            932075 : 'Financials',
        },
    },
    'us': {
        'file'         : 'US Industry Index.xlsx',
        'date_col'     : 'Name',
        'price_col'    : None,
        'format'       : 'wide',        # wide: one column per series
        'index_col'    : None,
        'benchmark_col': 'S&P 500 COMPOSITE - PRICE INDEX',
        # Verbose column header → GICS name
        'sector_map': {
            'S&P500 ES ENERGY - PRICE INDEX'                : 'Energy',
            'S&P500 ES MATERIALS - PRICE INDEX'             : 'Materials',
            'S&P500 ES INDUSTRIALS - PRICE INDEX'           : 'Industrials',
            'S&P500 ES CONSUMER DISCRETIONARY - PRICE INDEX': 'Consumer Discretionary',
            'S&P500 ES CONSUMER STAPLES - PRICE INDEX'      : 'Consumer Staples',
            'S&P500 ES HEALTH CARE - PRICE INDEX'           : 'Health Care',
            'S&P500 ES FINANCIALS - PRICE INDEX'            : 'Financials',
            'S&P500 ES REAL ESTATE - PRICE INDEX'           : 'Real Estate',
            'S&P500 ES INFO TECHNOLOGY - PRICE INDEX'       : 'Information Technology',
            'S&P500 ES COMM. SVS - PRICE INDEX'             : 'Communication Services',
            'S&P500 ES UTILITIES - PRICE INDEX'             : 'Utilities',
        },
    },
}

log.info('Config loaded. Markets=%s | HORIZON=%d | ROLLING_WINDOW=%d',
         MARKETS, HORIZON, ROLLING_WINDOW)

## 2.0 — Data Loading & Return Construction

### Code Explanation:

<span style="color:#63C2FF">2.1 Issue Logger:</span>

Append a timestamped diagnostic entry to the issue log and emit a warning.

Used throughout the pipeline (data-quality flags, ADF non-stationarity, GARCH/EGARCH convergence problems, rolling-window failures, etc.) to record non-fatal problems for later review without halting execution.

Args:
- market (str): Market identifier (e.g. 'china', 'us'), used to tag the log entry
- context (str): Short label identifying which pipeline stage raised the issue (e.g. 'JUMP', 'ADF', 'LAG_SELECTION', 'ROLLING')
- message (str): Human-readable description of the issue

Returns:
- None: nothing is returned, writes a line to ISSUE_LOG and emits a logger warning as a side effect

---

<span style="color:#63C2FF">2.2 Data Loader — load_price_panel(market):</span>

Loads raw price data and outputs a clean, unified DataFrame (DatetimeIndex named 'date', columns = the 11 canonical GICS sectors) regardless of which market is being processed.

The two markets arrive in structurally different formats, so the function branches:

- China (long format): 

    The raw file has one row per (Date, sector code) pair, where sector codes are integers like 986, 987. The CSI 300 benchmark code (300) is filtered out before pivoting, then the pivot reshapes rows → columns, and the numeric codes are renamed to readable GICS names.

- US (wide format): 

    Each sector is already its own column, with verbose names like 'S&P500 ES ENERGY - PRICE INDEX'. The benchmark column is dropped, then a simple rename maps headers to GICS names.

Any GICS sector genuinely missing from the raw data is added as a NaN column rather than crashing allowing the schema to stay consistent downstream.

---

<span style="color:#63C2FF">2.3 Return Constructor — compute_pct_returns(price_panel, market):</span>

Computes percentage log-returns: 

$$R_{i,t} = 100 \cdot \ln⁡(P_t/P_t − 1)$$

Zeros are replaced with NaN before taking the log (log(0) = −∞ would corrupt everything downstream).
.diff() computes the day-over-day log difference; multiplying by 100 rescales the series into percentage terms — this is important because GARCH optimizers are numerically more stable on values in the range ±5 than ±0.05.
Only rows where every sector is NaN are dropped (the first row, which has no prior price). Rows with partial NaNs are preserved so each sector's GARCH fit can use its own full available history.
Any return exceeding ±20% is flagged to the issue log as a potential data error or extreme event, but kept in the data.

---

<span style="color:#63C2FF">2.4 Main Loop:</span>

Iterates over both markets, calls the two functions above, stores results in all_prices and all_returns dictionaries (keyed by 'china'/'us'), and persists both to outputs/[market]/.

### Code Cell:

In [ ]:
# ── 2.1 Issue logger ──────────────────────────────────────────────────────────
def _log_issue(market: str, context: str, message: str) -> None:
    """Append a timestamped diagnostic line to ISSUE_LOG and emit a warning."""
    ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f'[{ts}] [{market.upper()}] [{context}] {message}\n'
    with open(ISSUE_LOG, 'a') as fh:
        fh.write(entry)
    log.warning(entry.strip())  


# ── 2.2 Data loader ───────────────────────────────────────────────────────────
def load_price_panel(market: str) -> pd.DataFrame:
    """
    Load and normalise raw price data for *market* into a clean
    date-indexed DataFrame with canonical GICS_SECTORS columns.

    China (long format)
    -------------------
    Raw file has one row per (Date, Index) pair where Index is a numeric
    sector code.  We pivot to wide format then rename codes → GICS names.
    The CSI 300 benchmark code is excluded before pivoting.

    US (wide format)
    ----------------
    Raw file has one column per series (sector + benchmark).  We rename
    the verbose column headers to GICS names and drop the benchmark.

    Returns
    -------
    pd.DataFrame  shape (T, 11), DatetimeIndex, columns = GICS_SECTORS
    """
    cfg      = MARKET_CONFIGS[market]
    filepath = DATA_DIR / market / cfg['file']

    # ── Resolve file path ─────────────────────────────────────────────────────
    if not filepath.exists():
        for ext in ['.xlsx', '.xls', '.csv']:
            alt = filepath.with_suffix(ext)
            if alt.exists():
                filepath = alt
                break
    if not filepath.exists():
        raise FileNotFoundError(f'[{market.upper()}] Raw data not found: {filepath}')

    # ── Read raw file ─────────────────────────────────────────────────────────
    if filepath.suffix in ('.xlsx', '.xls'):
        df_raw = pd.read_excel(filepath)
    else:
        df_raw = pd.read_csv(filepath)

    # ── Branch: long (China) vs wide (US) ────────────────────────────────────
    if cfg['format'] == 'long':
        df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
        df_raw.dropna(subset=[cfg['date_col']], inplace=True)

        # Exclude benchmark index before pivoting
        if cfg['benchmark_col'] is not None:
            df_raw = df_raw[df_raw[cfg['index_col']] != cfg['benchmark_col']]

        # Pivot: rows=Date, cols=numeric sector code
        price_panel = (
            df_raw
            .groupby([cfg['date_col'], cfg['index_col']])[cfg['price_col']]
            .last()
            .unstack()
        )
        # Rename numeric codes → GICS names (unknown codes are silently dropped)
        price_panel.rename(columns=cfg['sector_map'], inplace=True)

    else:  # wide (US)
        df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
        df_raw.set_index(cfg['date_col'], inplace=True)
        df_raw = df_raw[~df_raw.index.isna()]

        # Drop benchmark column before renaming so it never appears in output
        if cfg['benchmark_col'] and cfg['benchmark_col'] in df_raw.columns:
            df_raw.drop(columns=[cfg['benchmark_col']], inplace=True)

        price_panel = df_raw.rename(columns=cfg['sector_map'])

    # ── Final normalisation ───────────────────────────────────────────────────
    # Coerce everything to float
    for col in price_panel.columns:
        price_panel[col] = pd.to_numeric(price_panel[col], errors='coerce')

    # Name the index 'date' consistently
    price_panel.index.name = 'date'
    price_panel.index = pd.DatetimeIndex(price_panel.index)
    price_panel.sort_index(inplace=True)

    # Add any missing GICS columns as NaN (keeps schema unified even if data
    # is unavailable for one sector in one market)
    for sec in GICS_SECTORS:
        if sec not in price_panel.columns:
            price_panel[sec] = np.nan

    # Enforce canonical column order
    price_panel = price_panel[GICS_SECTORS]

    log.info('[%s] Price panel ready: %s  date range: %s → %s',
             market.upper(), price_panel.shape,
             price_panel.index[0].date(), price_panel.index[-1].date())
    return price_panel


# ── 2.3 Return constructor ────────────────────────────────────────────────────
def compute_pct_returns(price_panel: pd.DataFrame, market: str) -> pd.DataFrame:
    """
    R(i,t) = 100 * ln(P_t / P_{t-1})

    • Zero prices are treated as missing (ln(0) = −∞).
    • Only rows where every sector is NaN are dropped (the first-difference
      row).  Rows with partial NaNs survive so each sector's GARCH fit uses
      its own complete history.
    • Returns exceeding ±JUMP_THRESHOLD are flagged to ISSUE_LOG but kept.
    """
    log_prices  = np.log(price_panel.replace(0, np.nan))
    pct_returns = log_prices.diff() * 100.0
    pct_returns.dropna(how='all', inplace=True)

    # Quality audit
    jump_mask = pct_returns.abs() > JUMP_THRESHOLD
    n_flags   = int(jump_mask.sum().sum())
    if n_flags:
        for col in jump_mask.columns:
            flagged_dates = jump_mask.index[jump_mask[col]]
            for dt in flagged_dates:
                _log_issue(market, 'JUMP',
                           f'{col} {dt.date()} R={pct_returns.loc[dt, col]:.2f}%')

    n_nan = int(pct_returns.isna().sum().sum())
    log.info('[%s] Returns: %s obs | %d jump flags | %d NaNs',
             market.upper(), pct_returns.shape, n_flags, n_nan)
    return pct_returns

# ── 2.4 Main Loop ─────────────────────────────────────────────────────────────
all_prices  = {}
all_returns = {}

for MARKET in MARKETS:
    out_m = OUTPUTS_DIR / MARKET
    out_m.mkdir(parents=True, exist_ok=True)

    prices  = load_price_panel(MARKET)
    returns = compute_pct_returns(prices, MARKET)

    # Persist
    returns.to_csv(out_m / 'pct_returns.csv')
    prices.to_csv(out_m / 'price_panel.csv')

    all_prices[MARKET]  = prices
    all_returns[MARKET] = returns

    print(f'\n── [{MARKET.upper()}] Return Summary ──')
    display(returns.describe().round(4))

## 3.0 — GARCH(1,1)-t Conditional Volatility Panel

### Code Explanation:

<span style="color:#63C2FF">3.1 EXPECTED_BOUNDS & _check_garch_params():</span>

After fitting, every sector's estimated parameters are validated against typical empirical ranges for daily equity volatility. The key check is $persistence = \alpha + \beta$:

- $≥$ 1: explosive, variance never decays back to its mean, the model is non-stationary.

- $<$ 0.95: unusually low, volatility shocks fade too quickly to be realistic for equity data.

- $>$ 0.99: near-integrated, shocks fade very slowly, technically still stationary but borderline. Flagged as a warning, not a rejection.

A non-positive $\omega$ (the constant baseline variance) is also flagged since it makes the unconditional variance undefined. All flags are informational, the model is still used.

---

<span style="color:#63C2FF">3.2 _fit_garch(series, sector):</span>

Fits a constant-mean GARCH(1,1) with Student-t innovations to a single sector's return series. Two attempts are made: first with `MAX_ITER = 200` iterations, then `MAX_ITER_RETRY = 500` if the optimizer fails to converge on the first try. If both fail, the function returns `None` for the volatility series and logs the failure. The downstream panel fills that sector with `NaN` rather than crashing. On success it returns the conditional volatility series $\sigma_t$ and a parameter record dict.

---

<span style="color:#63C2FF">3.3 build_garch_panel(pct_returns, market):</span>

Loops over all 11 GICS sectors, calls `_fit_garch()` for each, and assembles the results into two DataFrames: the `volatility panel` (one column of $\sigma_t$ per sector) and a `parameter table` (one row of diagnostics per sector). Sectors that are entirely NaN in the input are skipped gracefully with a NaN column, so the output always has all 11 columns.
This GARCH volatility panel is the only input to the VAR stage. EGARCH is not fitted here since the symmetric GARCH(1,1) is the project-mandated model for the connectedness pipeline.

---

<span style="color:#63C2FF">3.4 Main Loop:</span>

Similar to previous sections, but used here to call current cell's new functions.

### Code Cell:

In [ ]:
# ── 3.1 ───────────────────────────────────────────────────────────────────────
EXPECTED_BOUNDS = {
    'omega': (1e-8, 10.),
    'alpha': (0.01, 0.25),
    'beta' : (0.70, 0.99),
}


def _check_garch_params(params: dict, sector: str) -> tuple[list, float]:
    """
    Validate GARCH(1,1) parameter estimates against economic bounds.

    Returns
    -------
    warns   : list[str]   human-readable flag strings (empty → all OK)
    persist : float       alpha + beta  (NaN if extraction failed)
    """
    alpha   = params.get('alpha[1]', params.get('alpha', np.nan))
    beta    = params.get('beta[1]',  params.get('beta',  np.nan))
    omega   = params.get('omega', np.nan)
    persist = (alpha + beta) if not (np.isnan(alpha) or np.isnan(beta)) else np.nan

    warns = []
    if not np.isnan(persist):
        if persist >= 1.0:
            warns.append(f'alpha+beta={persist:.4f} >= 1 (explosive)')
        elif persist < PERSISTENCE_LOW:
            warns.append(f'low persistence={persist:.4f}')
        elif persist > PERSISTENCE_HIGH:
            warns.append(f'high persistence={persist:.4f}')
    if not np.isnan(omega) and omega <= 0:
        warns.append(f'omega={omega:.2e} <= 0')

    return warns, persist

# ── 3.2 ───────────────────────────────────────────────────────────────────────
def _fit_garch(series: pd.Series, sector: str) -> tuple:
    """
    Fit a constant-mean GARCH(1,1)-t model to *series*.

    Two attempts are made: MAX_ITER iterations first, then MAX_ITER_RETRY.
    Returns (vol_series | None, param_dict, converged).
    """
    s = series.dropna()
    if len(s) < 100:
        return None, {'sector': sector, 'converged': False,
                      'note': 'insufficient data'}, False

    for attempt, max_it in enumerate([MAX_ITER, MAX_ITER_RETRY], 1):
        try:
            am  = arch_model(s, vol='Garch', p=1, q=1,
                             dist=INNOVATION_DIST, mean='Constant')
            res = am.fit(disp='off', show_warning=False,
                         options={'maxiter': max_it})
            conv = (res.convergence_flag == 0)
            if not conv and attempt == 1:
                continue   # retry with more iterations

            params         = res.params.to_dict()
            warns, persist = _check_garch_params(params, sector)
            alpha  = params.get('alpha[1]', np.nan)
            beta   = params.get('beta[1]',  np.nan)

            rec = {
                'sector'     : sector,
                'omega'      : round(params.get('omega', np.nan), 8),
                'alpha'      : round(alpha,   6),
                'beta'       : round(beta,    6),
                'persistence': round(persist, 6) if not np.isnan(persist) else np.nan,
                'nu'         : round(params.get('nu', np.nan), 4),
                'log_lik'    : round(res.loglikelihood, 4),
                'aic'        : round(res.aic, 4),
                'converged'  : conv,
                'warnings'   : '; '.join(warns) if warns else 'OK',
            }
            vol_s = pd.Series(res.conditional_volatility,
                              index=s.index, name=sector)
            return vol_s, rec, conv

        except Exception as exc:
            if attempt == 2:
                return None, {'sector': sector, 'converged': False,
                              'note': str(exc)}, False

    return None, {'sector': sector, 'converged': False,
                  'note': 'all attempts failed'}, False

# ── 3.3 ───────────────────────────────────────────────────────────────────────
def build_garch_panel(pct_returns: pd.DataFrame,
                      market: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fit GARCH(1,1)-t to every GICS sector and assemble the volatility panel.

    Parameters
    ----------
    pct_returns : pct log-returns, columns = GICS_SECTORS
    market      : used for logging

    Returns
    -------
    vol_panel   : pd.DataFrame  conditional volatility σ(t), cols = GICS_SECTORS
    param_table : pd.DataFrame  one row per sector with fit diagnostics
    """
    vols, params_list = {}, []

    available = [s for s in GICS_SECTORS if s in pct_returns.columns]

    for sec in available:
        if pct_returns[sec].isna().all():
            vols[sec] = pd.Series(np.nan, index=pct_returns.index, name=sec)
            params_list.append({'sector': sec, 'converged': False,
                                 'note': 'column is all NaN'})
            continue

        vol_s, rec, conv = _fit_garch(pct_returns[sec], sec)
        vols[sec] = vol_s if vol_s is not None else pd.Series(
            np.nan, index=pct_returns.index, name=sec)
        params_list.append(rec)
        log.info('[%s] GARCH  %-28s persist=%.4f  converged=%s',
                 market.upper(), sec, rec.get('persistence', np.nan),
                 rec.get('converged'))

    vol_panel   = pd.DataFrame(vols).sort_index()
    param_table = pd.DataFrame(params_list).set_index('sector')
    return vol_panel, param_table


# ── 3.4 Main Loop ─────────────────────────────────────────────────────────────
garch_panels = {}

for MARKET in MARKETS:
    print(f'\n{"="*50}\nGARCH VOLATILITY PANEL — {MARKET.upper()}\n{"="*50}')
    out_m = OUTPUTS_DIR / MARKET
    out_m.mkdir(parents=True, exist_ok=True)

    log.info('[%s] Fitting GARCH(1,1)-t panels…', MARKET.upper())
    g_panel, g_params = build_garch_panel(all_returns[MARKET], MARKET)

    # Persist
    g_panel.to_csv(out_m / 'garch_volatility_panel.csv')
    g_params.to_csv(out_m / 'garch_parameters.csv')

    garch_panels[MARKET] = g_panel

    print(f'\n── [{MARKET.upper()}] GARCH(1,1)-t Parameters ──')
    display(g_params)

## 4.0 — ADF Stationarity & VAR(p) Fitting

### Code Explanation:

<span style="color:#FF6363">Inputs  : </span>

- <span style="color:#40E0D0">arch_panels[market]</span>   —> GARCH(1,1) conditional volatilities ONLY

<span style="color:#FF6363">Outputs : </span>

- <span style="color:#40E0D0">var_results[market]</span>   —> fitted VAR objects

- <span style="color:#40E0D0">vol_panels[market]</span>    —> cleaned vol panel (NaN/Inf rows dropped)

- <span style="color:#40E0D0">adf_tables[market]</span>    —> ADF stationarity summary

<span style="color:#63C2FF">4.1 run_adf(vol_panel, market):</span>

Runs an `Augmented Dickey-Fuller` test on each of the 11 GARCH volatility series. The VAR model assumes all input series are stationary, i.e. they fluctuate around a stable mean rather than drifting upward or downward indefinitely. The ADF test formalises this: we are testing the null hypothesis that a series has a unit root (non-stationary). Rejection at $p < 0.05$ means we can safely use the series in a level VAR without differencing. Any sector that fails is flagged to the issue log.

---

<span style="color:#63C2FF">4.2 clean_vol_panel(vol_panel, market):</span>

Drops any row containing `NaN` or $\pm\infty$ before passing the panel to statsmodels' VAR. This is a hard requirement: VAR fitting uses LAPACK matrix operations internally, and a single NaN in the covariance matrix causes a low-level DLASCL error. In practice only a handful of rows (typically the first 1-3 observations of each GARCH series before the filter warms up) are affected.

---

<span style="color:#63C2FF">4.3 fit_var(vol_panel, lag_order, market):</span>

Fits the VAR($p$) model on the cleaned volatility panel. The lag order is fixed, $p=2$ for China, $p=8$ for the US, based on AIC pre-selection from a prior run. The `select_order()` call is still executed and printed for reference, letting you verify the fixed orders were indeed AIC-optimal, but its output does not change what goes into `fit()`.
The resulting VAR captures the interdependencies between all 11 sectors simultaneously, i.e. given today's volatility in Energy, Materials, Financials, etc., `what does the model predict about tomorrow's volatilities across the whole system?` That system of predictions is what GFEVD then decomposes to quantify spillovers.

---

<span style="color:#63C2FF">4.4 Main Loop:</span>

Similar to previous sections, but used here to call current cell's new functions.

### Code Cell:

In [ ]:
# ── 4.1 ───────────────────────────────────────────────────────────────────────
def run_adf(vol_panel: pd.DataFrame, market: str) -> pd.DataFrame:
    """
    ADF unit-root test on each sector's GARCH volatility series.

    H0: series has a unit root (non-stationary).
    Rejecting H0 (p < ADF_PVAL_THR) supports use in a level VAR.

    Returns
    -------
    adf_table : pd.DataFrame  indexed by sector, columns [adf_stat, p_value, stationary]
    """
    records = []
    for sec in vol_panel.columns:
        s = vol_panel[sec].dropna()
        try:
            stat, pval, *_ = adfuller(s, maxlag=ADF_MAXLAG, autolag='AIC')
            stationary = bool(pval < ADF_PVAL_THR)
        except Exception as e:
            stat, pval, stationary = np.nan, np.nan, False
            _log_issue(market, 'ADF', f'{sec}: {e}')

        if not stationary:
            _log_issue(market, 'ADF',
                       f'{sec} p={pval:.4f} — unit root not rejected')
        records.append({
            'sector'    : sec,
            'adf_stat'  : round(stat,  4) if not np.isnan(stat)  else np.nan,
            'p_value'   : round(pval,  4) if not np.isnan(pval)  else np.nan,
            'stationary': stationary,
        })

    return pd.DataFrame(records).set_index('sector')


# ── 4.2 ───────────────────────────────────────────────────────────────────────
def clean_vol_panel(vol_panel: pd.DataFrame, market: str) -> pd.DataFrame:
    """
    Drop any row containing NaN or ±Inf before passing to VAR.
    LAPACK (used internally by statsmodels) will error on such values.
    """
    n_before = len(vol_panel)
    cleaned  = vol_panel.replace([np.inf, -np.inf], np.nan).dropna()
    n_dropped = n_before - len(cleaned)
    if n_dropped:
        log.info('[%s] Volatility panel: dropped %d rows with NaN/Inf '
                 '(%d → %d obs)', market.upper(), n_dropped, n_before, len(cleaned))
    return cleaned


# ── 4.3 ───────────────────────────────────────────────────────────────────────
def fit_var(vol_panel: pd.DataFrame, lag_order: int,
            market: str) -> object:
    """
    Fit VAR(lag_order) on the cleaned volatility panel.

    The lag order is fixed (not re-selected by AIC here) to enforce the
    project specification:  China p=2, US p=8.  The AIC lag-selection
    table is printed for diagnostic reference only.

    Returns
    -------
    var_result : statsmodels VARResultsWrapper
    """
    # Print AIC selection table for reference (does not change lag_order)
    try:
        sel = VAR(vol_panel).select_order(maxlags=MAX_LAGS)
        log.info('[%s] AIC-optimal lag (reference): p=%d  |  Fixed lag used: p=%d',
                 market.upper(), int(sel.aic), lag_order)
        print(sel.summary())
    except Exception as e:
        _log_issue(market, 'LAG_SELECTION',
                   f'AIC selection failed (informational): {e}')

    var_result = VAR(vol_panel).fit(maxlags=lag_order, ic=None)
    log.info('[%s] VAR(%d) fitted.  Log-likelihood: %.2f',
             market.upper(), lag_order, var_result.llf)
    return var_result


# ── 4.4 Main Loop ─────────────────────────────────────────────────────────────
vol_panels  = {}
var_results = {}
adf_tables  = {}

for MARKET in MARKETS:
    print(f'\n{"="*55}\nADF + VAR FITTING — {MARKET.upper()}\n{"="*55}')
    out_m    = OUTPUTS_DIR / MARKET

    # Input: GARCH(1,1) conditional volatilities only
    raw_vol  = garch_panels[MARKET].copy()
    lag_p    = VAR_LAG_ORDERS[MARKET]

    # ADF
    adf_tab  = run_adf(raw_vol, MARKET)
    print(f'\n── [{MARKET.upper()}] ADF Stationarity ──')
    display(adf_tab)

    # Clean
    vol_clean = clean_vol_panel(raw_vol, MARKET)

    # VAR
    var_res   = fit_var(vol_clean, lag_p, MARKET)

    # Persist
    adf_tab.to_csv(out_m / 'adf_stationarity.csv')
    with open(out_m / 'var_result.pkl', 'wb') as fh:
        pickle.dump(var_res, fh)
    with open(out_m / 'var_lag_order.txt', 'w') as fh:
        fh.write(
            f'Market: {MARKET.upper()}\n'
            f'Lag order (fixed): {lag_p}\n'
            f'Timestamp: {datetime.now().isoformat()}\n'
        )

    vol_panels[MARKET]  = vol_clean
    var_results[MARKET] = var_res
    adf_tables[MARKET]  = adf_tab

    print(f'\nVAR({lag_p}) fit complete  [{MARKET.upper()}]')

## 5.0 — GFEVD Static Connectedness & Adjacency Export

### Code Explanation:

<span style="color:#63C2FF">5.1 compute_gfevd(var_res, H):</span>

Implements the KPPS Generalised FEVD formula. For each pair of sectors $(i, j)$, it calculates what fraction of sector $i$'s forecast error variance, over the next $H=10$ days, is attributable to shocks originating in sector $j$:

$$\tilde{\theta}_{ij}(H) = \frac{\sigma_{jj}^{-1} \sum_{h=0}^{H} (e_i' A_h \Sigma e_j)^2}{\sum_{h=0}^{H} (e_i' A_h \Sigma A_h' e_i)}$$

$A_h$ are the impulse-response matrices at horizon $h$ (how the system responds $h$ days after a shock), and $\Sigma$ is the full residual covariance matrix. Using $\Sigma$ rather than a Cholesky factor makes the result order-invariant. Hence, the spillover from Energy to Financials is the same regardless of how you order the sectors in the VAR.

`np.asarray()` is called on the IRF output to strip Pandas index labels, which would otherwise cause `InvalidIndexError` when using integer positional indexing (`A[h, i, :]`).

---

<span style="color:#63C2FF">5.2 row_normalise(theta):</span>

Because the KPPS raw GFEVD rows do not generally sum to 1 (unlike Cholesky-orthogonalised FEVD), explicit row normalisation is a mandatory step before computing any directional measures. Each entry $$\tilde{\theta}^*_{ij} = \tilde{\theta}_{ij} / \sum_j \tilde{\theta}_{ij}$$ now represents a genuine share: "X% of sector $i$'s forecast error comes from sector $j$."

---

<span style="color:#63C2FF">5.3 directional_measures(theta_star, names):</span>

Computes the three core Diebold-Yilmaz network metrics from the row-normalised matrix (all expressed as percentages):

- $TO_j$​: how much sector $j$ transmits to all others, column sum of the off-diagonal entries.

- $FROM_i$​: how much sector $i$ absorbs from all others, row sum of the off-diagonal entries.

- $NET_i$​ = TO − FROM: 

    positive = net transmitter (spreads more risk than it receives)
    
    negative = net receiver (absorbs more than it sends).

TSI (Total Spillover Index): mean of all off-diagonal entries × 100. A single number summarising how interconnected the system is.


---

<span style="color:#63C2FF">5.4 verify_baseline(measures, tsi, market):</span>

Prints a `PASS/WARN` check against the project's pre-specified verification targets (China TSI ≈ 81.2%, Financials NET ≈ −16; US TSI ≈ 80.5%, Financials NET ≈ +28). Tolerances are ±2% on TSI and ±5 on NET. A WARN here means either the data, the lag order, or the GFEVD implementation needs to be re-examined.

---

<span style="color:#63C2FF">5.5 save_adjacency(theta_star, names, out_dir, tag):</span>

Saves the row-normalised connectedness matrix in two formats for the GNN stage:

- Index-less CSV (no row/column labels, raw floats): directly loadable as a NumPy array.

- .npy binary: faster to load and byte-exact.

The adjacency matrix is a weighted directed graph: entry $(i, j)$ is the edge weight from sector $j$ to sector $i$, representing the share of $i$'s forecast error variance explained by $j$.

---

<span style="color:#63C2FF">5.6 Main Loop:</span>

Similar to previous sections, but used here to call current cell's new functions.

### Code Cell:

In [ ]:
# ── 5.1 ───────────────────────────────────────────────────────────────────────
def compute_gfevd(var_res, H: int) -> np.ndarray:
    """
    Generalised Forecast Error Variance Decomposition (KPPS).

    Uses the full residual covariance Σ (not the Cholesky factor), so
    sector ordering is irrelevant.

    θ̃(H)_{ij} = σ_jj⁻¹ · Σ_{h=0}^{H} (e_i' A_h Σ e_j)²
                 ─────────────────────────────────────────
                    Σ_{h=0}^{H} (e_i' A_h Σ A_h' e_i)

    The outer loop normalises each row to sum to 1 (see row_normalise).

    Parameters
    ----------
    var_res : statsmodels VARResultsWrapper
    H       : forecast horizon (days)

    Returns
    -------
    theta : np.ndarray  shape (n, n), raw (pre-normalisation) GFEVD
    """
    # np.asarray strips any Pandas index that would cause InvalidIndexError
    A   = np.asarray(var_res.irf(periods=H).irfs)   # shape (H+1, n, n)
    Sig = np.asarray(var_res.sigma_u)                # shape (n, n)
    n   = Sig.shape[0]
    sig_diag = np.diag(Sig)                          # σ_jj for j = 1..n

    theta = np.zeros((n, n))
    for i in range(n):
        # Denominator: scalar — total forecast-error variance of variable i
        denom = sum(
            A[h, i, :] @ Sig @ A[h, i, :]
            for h in range(H + 1)
        )
        for j in range(n):
            # Numerator: share of variable-j shocks in variable-i's FEVD
            numer = (1.0 / sig_diag[j]) * sum(
                (A[h, i, :] @ Sig[:, j]) ** 2
                for h in range(H + 1)
            )
            theta[i, j] = numer / denom if denom > 1e-10 else 0.0

    return theta


# ── 5.2 ───────────────────────────────────────────────────────────────────────
def row_normalise(theta: np.ndarray) -> np.ndarray:
    """
    Explicitly normalise every row of theta to sum to 1.
    Required by the KPPS GFEVD (raw rows do not generally sum to unity).
    """
    row_sums = theta.sum(axis=1, keepdims=True)
    return theta / np.where(row_sums == 0, 1.0, row_sums)


# ── 5.3 ───────────────────────────────────────────────────────────────────────
def directional_measures(theta_star: np.ndarray,
                          names: list) -> tuple[pd.DataFrame, float]:
    """
    Compute Diebold-Yilmaz TO, FROM, NET spillover measures.

    Parameters
    ----------
    theta_star : row-normalised GFEVD  (n × n)
    names      : sector name list (length n)

    Returns
    -------
    measures : pd.DataFrame  columns [TO, FROM, NET, OWN] (in %)
    tsi      : float         Total Spillover Index (%)
    """
    off = theta_star.copy()
    np.fill_diagonal(off, 0.0)

    TO   = off.sum(axis=0)   # column sums = spillovers TO sector j
    FROM = off.sum(axis=1)   # row    sums = spillovers FROM sector i
    NET  = TO - FROM
    OWN  = np.diag(theta_star)

    # TSI = mean off-diagonal share (× 100 for %)
    tsi  = off.sum() / len(names) * 100.0

    df = pd.DataFrame(
        {'TO': TO * 100, 'FROM': FROM * 100,
         'NET': NET * 100, 'OWN': OWN * 100},
        index=names,
    ).round(4)
    return df, tsi


# ── 5.4 ───────────────────────────────────────────────────────────────────────
def verify_baseline(measures: pd.DataFrame, tsi: float,
                    market: str) -> None:
    """
    Print a pass/warn check against the project verification baselines.

    Baselines:
      China : TSI ≈ 81.2 %  |  Financials NET ≈ −16
      US    : TSI ≈ 80.5 %  |  Financials NET ≈ +28
    """
    baselines = {
        'china': {'tsi': 81.2, 'fin_net': -16.0, 'tsi_tol': 2.0, 'net_tol': 5.0},
        'us'   : {'tsi': 80.5, 'fin_net': +28.0, 'tsi_tol': 2.0, 'net_tol': 5.0},
    }
    if market not in baselines:
        return
    b      = baselines[market]
    fin_net = measures.loc['Financials', 'NET'] if 'Financials' in measures.index else np.nan

    tsi_ok  = abs(tsi      - b['tsi'])     <= b['tsi_tol']
    net_ok  = abs(fin_net  - b['fin_net']) <= b['net_tol']
    status  = 'PASS' if (tsi_ok and net_ok) else 'WARN'

    print(f'\n── [{market.upper()}] Baseline Verification [{status}] ──')
    print(f"  TSI        : {tsi:.2f}%  (baseline ≈ {b['tsi']:.1f}%)  "
          f"Δ={abs(tsi - b['tsi']):.2f}  {'✓' if tsi_ok else '✗'}")
    print(f"  Financials NET: {fin_net:.2f}  (baseline ≈ {b['fin_net']:+.0f})  "
          f"Δ={abs(fin_net - b['fin_net']):.2f}  {'✓' if net_ok else '✗'}")


# ── 5.5 ───────────────────────────────────────────────────────────────────────
def save_adjacency(theta_star: np.ndarray,
                   names: list,
                   out_dir: Path,
                   tag: str = 'fullsample') -> None:
    """
    Persist the row-normalised connectedness matrix (adjacency graph) for
    downstream GNN consumption.

    Files written
    -------------
    connectedness_matrix_{tag}.csv   — index-less, clean float matrix
    connectedness_matrix_{tag}.npy   — NumPy binary (shape n×n)
    """
    csv_path = out_dir / f'connectedness_matrix_{tag}.csv'
    npy_path = out_dir / f'connectedness_matrix_{tag}.npy'

    # Index-less CSV (GNNs typically consume raw arrays)
    pd.DataFrame(theta_star, index=names, columns=names).to_csv(
        csv_path, index=False
    )
    np.save(npy_path, theta_star)
    log.info('Adjacency saved → %s  (shape %s)', csv_path.name, theta_star.shape)


# ── 5.6 Main Loop ─────────────────────────────────────────────────────────────
static_measures = {}
static_tsi      = {}
theta_stars     = {}

for MARKET in MARKETS:
    print(f'\n{"="*60}\nGFEVD STATIC CONNECTEDNESS — {MARKET.upper()}\n{"="*60}')
    out_m        = OUTPUTS_DIR / MARKET
    var_res      = var_results[MARKET]
    sector_names = list(vol_panels[MARKET].columns)

    theta_raw  = compute_gfevd(var_res, HORIZON)
    theta_star = row_normalise(theta_raw)
    measures, tsi = directional_measures(theta_star, sector_names)

    print(f'\nTotal Spillover Index (H={HORIZON}, full sample): {tsi:.2f}%')
    display(measures)

    verify_baseline(measures, tsi, MARKET)

    # Persist
    measures.to_csv(out_m / 'directional_measures.csv')
    save_adjacency(theta_star, sector_names, out_m, tag='fullsample')
    with open(out_m / 'tsi_fullsample.txt', 'w') as fh:
        fh.write(
            f'Market: {MARKET.upper()}\n'
            f'Lag order: {VAR_LAG_ORDERS[MARKET]}\n'
            f'TSI (H={HORIZON}): {tsi:.4f}%\n'
            f'Timestamp: {datetime.now().isoformat()}\n'
        )

    static_measures[MARKET] = measures
    static_tsi[MARKET]      = tsi
    theta_stars[MARKET]     = theta_star

## 6.0 — Rolling Connectedness (200-day window)

### Code Explanation:

<span style="color:#FF6363">Output files per market:</span>

- <span style="color:#40E0D0">rolling_tsi.csv</span>           —> date-indexed TSI + NET/TO/FROM per sector

- <span style="color:#40E0D0">adjacency/adj_{date}.npy</span>  —> (n×n) row-normalised connectedness per window

- <span style="color:#40E0D0">adjacency/adj_{date}.csv</span>  —> same, index-less (GNN-ready)

---

<span style="color:#63C2FF">6.1 rolling_connectedness(vol_panel, lag_order, market, out_dir):</span>

Implements a walk-forward estimation loop. Starting from day 200, at each step the function:

1. Slices the trailing 200-day window from the volatility panel.
2. Re-fits the VAR with the fixed lag order on that slice only.
3. Computes GFEVD and extracts TSI + all NET/TO/FROM values.
4. Saves the $(11 \times 11)$ adjacency matrix for that window's end date as both `.npy` and a header-free `.csv` into an `adjacency/` subfolder.
5. Appends the TSI and sector metrics to a running record.

The window then slides forward by 1 trading day (`ROLLING_STEP=1`), dropping the oldest observation and adding the newest.

The result is a `daily time series` of how system-wide connectedness evolves (e.g. TSI rising during the 2015 China crash or 2020 COVID shock) and a `temporal sequence of adjacency matrices` that the GNN will use as `graph inputs` over time. Failed windows (e.g. a near-singular covariance in a volatile stretch) are logged and skipped rather than halting the loop.

---

<span style="color:#63C2FF">6.2 Main Loop:</span>

Similar to previous sections, but used here to call current cell's new functions.

### Code Cells:

In [ ]:
# ── 6.1 ───────────────────────────────────────────────────────────────────────
def rolling_connectedness(vol_panel: pd.DataFrame,
                          lag_order: int,
                          market: str,
                          out_dir: Path) -> pd.DataFrame:
    """
    Walk-forward rolling GFEVD connectedness analysis.

    Parameters
    ----------
    vol_panel  : cleaned GARCH volatility panel (T × n)
    lag_order  : fixed VAR lag order for this market
    market     : string label for logging
    out_dir    : market output directory (adjacency/ sub-dir created inside)

    Returns
    -------
    rolling_df : pd.DataFrame  DatetimeIndex, columns [TSI, NET_*, TO_*, FROM_*]
    """
    adj_dir = out_dir / 'adjacency'
    adj_dir.mkdir(parents=True, exist_ok=True)

    dates        = vol_panel.index
    n_obs        = len(dates)
    sector_names = list(vol_panel.columns)
    records      = []
    n_ok = n_fail = 0

    log.info('[%s] Rolling: window=%d  step=%d  lag=%d  H=%d  n_obs=%d',
             market.upper(), ROLLING_WINDOW, ROLLING_STEP,
             lag_order, HORIZON, n_obs)

    for end_idx in range(ROLLING_WINDOW - 1, n_obs, ROLLING_STEP):
        s_idx  = end_idx - ROLLING_WINDOW + 1
        s_date = dates[s_idx]
        e_date = dates[end_idx]
        win    = vol_panel.iloc[s_idx: end_idx + 1]

        if len(win) < ROLLING_WINDOW:
            continue

        try:
            vr           = VAR(win).fit(maxlags=lag_order, ic=None)
            theta_raw    = compute_gfevd(vr, HORIZON)
            theta_star_w = row_normalise(theta_raw)
            meas, tsi_w  = directional_measures(theta_star_w, sector_names)

            # ── GNN adjacency snapshot ────────────────────────────────────────
            date_str = e_date.strftime('%Y%m%d')
            np.save(adj_dir / f'adj_{date_str}.npy', theta_star_w)
            pd.DataFrame(theta_star_w).to_csv(
                adj_dir / f'adj_{date_str}.csv', index=False, header=False
            )

            # ── Build TSI record ──────────────────────────────────────────────
            row = {'date': e_date, 'TSI': round(tsi_w, 4)}
            for sec in sector_names:
                row[f'NET_{sec}']  = round(meas.loc[sec, 'NET'],  4)
                row[f'TO_{sec}']   = round(meas.loc[sec, 'TO'],   4)
                row[f'FROM_{sec}'] = round(meas.loc[sec, 'FROM'], 4)
            records.append(row)
            n_ok += 1

        except Exception as exc:
            n_fail += 1
            _log_issue(market, 'ROLLING',
                       f'{s_date.date()}→{e_date.date()}: {exc}')

        if n_ok % 100 == 0 and n_ok:
            log.info('[%s] Rolling: %d windows done  (end=%s)',
                     market.upper(), n_ok, e_date.date())

    log.info('[%s] Rolling complete: %d ok  %d failed', market.upper(), n_ok, n_fail)

    if not records:
        log.warning('[%s] No rolling windows produced output.', market.upper())
        return pd.DataFrame()

    rolling_df = pd.DataFrame(records).set_index('date')
    rolling_df.index = pd.DatetimeIndex(rolling_df.index)
    return rolling_df


# ── 6.2 Main Loop ─────────────────────────────────────────────────────────────
rolling_results = {}

for MARKET in MARKETS:
    print(f'\n{"="*60}\nROLLING CONNECTEDNESS — {MARKET.upper()}\n{"="*60}')
    out_m     = OUTPUTS_DIR / MARKET
    lag_p     = VAR_LAG_ORDERS[MARKET]

    rolling_df = rolling_connectedness(
        vol_panels[MARKET], lag_p, MARKET, out_m
    )

    if not rolling_df.empty:
        rolling_df.to_csv(out_m / 'rolling_tsi.csv')
        rolling_results[MARKET] = rolling_df
        print(f'\nRolling TSI: {len(rolling_df)} windows saved.')
        print(f'Adjacency matrices → {out_m / "adjacency"}')
        display(rolling_df[['TSI']].tail(10))
    else:
        print(f'[{MARKET.upper()}] No rolling windows produced output.')

## Cell 7 — Robustness Check: Common Lag Orders

### Code Explanation:

<span style="color:#FF6363">Outputs:</span>

For each (market, lag) combination:
    
- <span style="color:#40E0D0">outputs/{market}/robustness/measures_p{lag}.csv</span>
    
- <span style="color:#40E0D0">outputs/{market}/robustness/tsi_p{lag}.txt</span>
    
- <span style="color:#40E0D0">outputs/{market}/robustness/connectedness_matrix_p{lag}.csv</span>
    
- <span style="color:#40E0D0">outputs/{market}/robustness/connectedness_matrix_p{lag}.npy</span>

A summary comparison table is printed for visual inspection.

---

<span style="color:#63C2FF">7.1 robustness_var_gfevd(vol_panel, lag_order, market):</span>

A thin wrapper that re-runs the full VAR → GFEVD pipeline for a given alternative lag order. Returns the directional measures table, TSI, and the raw adjacency matrix.

---

<span style="color:#63C2FF">7.2 rank_net_transmitters(measures):</span>

Sorts sectors by NET spillover in descending order (largest transmitters first). This ranked list is what gets compared across lag specifications by asking: does the sector ranking stay stable even if the precise NET values shift slightly?

---

<span style="color:#63C2FF">7.3 Main Loop:</span>

For each market and each lag order in `ROBUSTNESS_LAGS = [4, 8]`, the VAR/GFEVD is re-estimated on the full sample and the NET ranking is compared to the baseline (China $p=2$, US $p=8$) using `Spearman rank correlation` $\rho$. A $\rho$ close to 1.0 means the transmitter/receiver ordering is structurally stable, i.e. the conclusion doesn't depend on the specific lag chosen. This is a standard robustness requirement in the DY literature before reporting results.

All alternative-lag measures and adjacency matrices are saved under `outputs/{market}/robustness/`, and a consolidated summary table (`robustness_summary.csv`) is printed and written to the root outputs/ directory.

### Code Cell:

In [ ]:
# ── 7.1 ───────────────────────────────────────────────────────────────────────
def robustness_var_gfevd(vol_panel: pd.DataFrame,
                         lag_order: int,
                         market: str) -> tuple[pd.DataFrame, float]:
    """
    Fit VAR(lag_order) on *vol_panel* and compute full-sample GFEVD measures.

    Returns
    -------
    measures : pd.DataFrame  [TO, FROM, NET, OWN] in %
    tsi      : float         Total Spillover Index (%)
    """
    sector_names = list(vol_panel.columns)
    var_res      = VAR(vol_panel).fit(maxlags=lag_order, ic=None)
    theta_raw    = compute_gfevd(var_res, HORIZON)
    theta_star   = row_normalise(theta_raw)
    measures, tsi = directional_measures(theta_star, sector_names)
    return measures, tsi, theta_star

# ── 7.2 ───────────────────────────────────────────────────────────────────────
def rank_net_transmitters(measures: pd.DataFrame) -> pd.Series:
    """Return sectors sorted descending by NET (transmitters first)."""
    return measures['NET'].sort_values(ascending=False)


# ── 7.3.1 Main Loop ───────────────────────────────────────────────────────────
robustness_results = {}   # {(market, lag): (measures, tsi)}

print(f'\n{"="*65}\nROBUSTNESS CHECK — COMMON LAG ORDERS: {ROBUSTNESS_LAGS}\n{"="*65}')

for MARKET in MARKETS:
    out_m   = OUTPUTS_DIR / MARKET
    rob_dir = out_m / 'robustness'
    rob_dir.mkdir(parents=True, exist_ok=True)

    sector_names = list(vol_panels[MARKET].columns)
    baseline_rank = rank_net_transmitters(static_measures[MARKET])

    print(f'\n── Baseline rank [{MARKET.upper()}] (p={VAR_LAG_ORDERS[MARKET]}) ──')
    print(baseline_rank.to_string())

    for lag_p in ROBUSTNESS_LAGS:
        try:
            measures_r, tsi_r, theta_star_r = robustness_var_gfevd(
                vol_panels[MARKET], lag_p, MARKET
            )

            # Rank correlation with baseline (Spearman)
            rank_r  = rank_net_transmitters(measures_r)
            # Align on common index before rank-correlating
            common  = baseline_rank.index.intersection(rank_r.index)
            rho     = baseline_rank[common].rank().corr(
                rank_r[common].rank(), method='spearman'
            )

            print(f'\n  [{MARKET.upper()}] p={lag_p}  TSI={tsi_r:.2f}%  '
                  f'Rank-ρ vs baseline={rho:.4f}')
            display(measures_r[['NET']].sort_values('NET', ascending=False))

            # Persist
            measures_r.to_csv(rob_dir / f'measures_p{lag_p}.csv')
            save_adjacency(theta_star_r, sector_names, rob_dir,
                           tag=f'p{lag_p}')
            with open(rob_dir / f'tsi_p{lag_p}.txt', 'w') as fh:
                fh.write(
                    f'Market: {MARKET.upper()}\n'
                    f'Lag order: {lag_p}\n'
                    f'TSI (H={HORIZON}): {tsi_r:.4f}%\n'
                    f'Spearman rank-rho vs baseline: {rho:.4f}\n'
                    f'Timestamp: {datetime.now().isoformat()}\n'
                )

            robustness_results[(MARKET, lag_p)] = {
                'measures': measures_r,
                'tsi'     : tsi_r,
                'rank_rho': rho,
            }

        except Exception as exc:
            _log_issue(MARKET, f'ROBUSTNESS_p{lag_p}', str(exc))
            print(f'  [{MARKET.upper()}] p={lag_p} FAILED: {exc}')


# ── 7.3.2 Main Loop ───────────────────────────────────────────────────────────
# ── Summary comparison table ──────────────────────────────────────────────────
print(f'\n{"="*65}\nROBUSTNESS SUMMARY\n{"="*65}')
summary_rows = []
for (mkt, lag_p), res in robustness_results.items():
    summary_rows.append({
        'Market'    : mkt.upper(),
        'Lag'       : lag_p,
        'TSI (%)'   : round(res['tsi'], 2),
        'Rank-ρ'    : round(res['rank_rho'], 4),
        'Fin NET'   : round(res['measures'].loc['Financials', 'NET'], 2)
                      if 'Financials' in res['measures'].index else np.nan,
    })

# ── 7.3.3 Main Loop ───────────────────────────────────────────────────────────
# ── Add baselines ─────────────────────────────────────────────────────────────
for mkt in MARKETS:
    base_fin = (static_measures[mkt].loc['Financials', 'NET']
                if 'Financials' in static_measures[mkt].index else np.nan)
    summary_rows.insert(
        0 if mkt == 'china' else
        len([r for r in summary_rows if r['Market'] == 'CHINA']),
        {
            'Market' : mkt.upper(),
            'Lag'    : VAR_LAG_ORDERS[mkt],
            'TSI (%)': round(static_tsi[mkt], 2),
            'Rank-ρ' : 1.0,
            'Fin NET': round(base_fin, 2),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index(['Market', 'Lag'])
display(summary_df)
summary_df.to_csv(OUTPUTS_DIR / 'robustness_summary.csv')